# Benchmarks: Standardized Tests for AI Models

## What You'll Learn

In this notebook, we'll explore **AI benchmarks** -- the standardized tests that
researchers use to compare models. We'll:

1. Understand what benchmarks are and why they matter
2. Load and explore real benchmark datasets
3. See what benchmark questions look like
4. Run a simple evaluation on a pre-trained model
5. Learn about leaderboards and how to interpret scores

### The One-Sentence Summary

> **Benchmarks are standardized exams for AI.** Just like SATs let you compare
> students across schools, benchmarks let you compare AI models fairly.

---
## 1. Setup

In [ ]:
# Install if needed (uncomment the lines below)
# !pip install datasets transformers torch matplotlib

import matplotlib.pyplot as plt
import random

print("Basic setup done!")

---
## 2. What Is a Benchmark?

A benchmark has three parts:

```
┌─────────────────────────────────────────────────────────┐
│                 Anatomy of a Benchmark                   │
│                                                          │
│  1. QUESTIONS/TASKS                                      │
│     A set of standardized questions or tasks              │
│     (e.g., 16,000 multiple choice questions)             │
│                                                          │
│  2. CORRECT ANSWERS                                      │
│     The ground truth for grading                         │
│     (e.g., the right answer for each question)           │
│                                                          │
│  3. SCORING METHOD                                       │
│     How to compute the final score                       │
│     (e.g., % correct, F1, exact match, etc.)             │
└─────────────────────────────────────────────────────────┘
```

Let's load a real benchmark and see what it looks like!

---
## 3. Exploring MMLU: The "College Entrance Exam" for AI

**MMLU (Massive Multitask Language Understanding)** tests knowledge across
57 subjects. Let's load it and look at some questions.

In [ ]:
from datasets import load_dataset

# Load a subset of MMLU (we'll use a few subjects)
print("Loading MMLU dataset (this may take a moment)...")

# Load specific subjects
subjects = {
    "high_school_mathematics": "High School Math",
    "high_school_physics": "High School Physics",
    "high_school_us_history": "US History",
    "computer_science": "Computer Science",
}

mmlu_data = {}
for subject_id, subject_name in subjects.items():
    try:
        ds = load_dataset("cais/mmlu", subject_id, split="test", trust_remote_code=True)
        mmlu_data[subject_name] = ds
        print(f"  Loaded {subject_name}: {len(ds)} questions")
    except Exception as e:
        print(f"  Could not load {subject_name}: {e}")

print(f"\nLoaded {sum(len(v) for v in mmlu_data.values())} questions total!")

In [ ]:
# Let's look at some actual MMLU questions!
answer_labels = ["A", "B", "C", "D"]

print("Sample MMLU Questions")
print("=" * 70)

for subject_name, dataset in mmlu_data.items():
    # Pick a random question
    idx = random.randint(0, len(dataset) - 1)
    q = dataset[idx]

    print(f"\nSubject: {subject_name}")
    print(f"Question: {q['question']}")
    for i, choice in enumerate(q['choices']):
        marker = "-->" if i == q['answer'] else "   "
        print(f"  {marker} {answer_labels[i]}) {choice}")
    print(f"Correct Answer: {answer_labels[q['answer']]}")
    print("-" * 70)

---
## 4. Building a Simple Benchmark Evaluator

Let's create a simple system that:
1. Takes a model's answers
2. Compares them to the correct answers
3. Computes the accuracy score

We'll start with a "dummy" model (random guessing) to set a baseline,
then try a real model.

In [ ]:
def evaluate_on_mmlu(model_answers, correct_answers):
    """
    Evaluate a model's answers against the correct answers.

    Args:
        model_answers: list of ints (0-3, representing A-D)
        correct_answers: list of ints (0-3)

    Returns:
        dict with accuracy and per-question results
    """
    correct = sum(1 for m, c in zip(model_answers, correct_answers) if m == c)
    total = len(correct_answers)
    accuracy = correct / total if total > 0 else 0

    return {
        "correct": correct,
        "total": total,
        "accuracy": accuracy,
    }


# BASELINE: Random guessing model
# With 4 choices (A, B, C, D), random guessing should get ~25%
print("Baseline: Random Guessing Model")
print("=" * 50)

for subject_name, dataset in mmlu_data.items():
    correct_answers = [q["answer"] for q in dataset]
    random_answers = [random.randint(0, 3) for _ in correct_answers]

    result = evaluate_on_mmlu(random_answers, correct_answers)
    print(f"  {subject_name:25s}: {result['accuracy']:.1%} ({result['correct']}/{result['total']})")

print(f"\nExpected: ~25% (random chance with 4 choices)")

In [ ]:
# "ALWAYS A" model -- another bad strategy, but shows why we need benchmarks
print("Another Bad Model: Always Answers 'A'")
print("=" * 50)

for subject_name, dataset in mmlu_data.items():
    correct_answers = [q["answer"] for q in dataset]
    always_a = [0] * len(correct_answers)  # Always pick A

    result = evaluate_on_mmlu(always_a, correct_answers)
    print(f"  {subject_name:25s}: {result['accuracy']:.1%}")

print(f"\nAlso ~25% -- because answer choices are roughly equally distributed.")
print(f"This shows that BOTH random and 'always A' are equally useless!")

---
## 5. Evaluating a Real Model

Now let's evaluate a real language model on some MMLU questions.
We'll use a small model (GPT-2) so it runs on any computer.

> **Note:** GPT-2 is an older, small model -- it won't score well on MMLU.
> Modern models like GPT-4 or Claude score 85%+. But this demonstrates the process.

In [ ]:
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer

print("Loading GPT-2 model...")
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")
model.eval()
print("Model loaded!")

In [ ]:
def gpt2_answer_mmlu(question, choices):
    """
    Use GPT-2 to answer an MMLU question by computing the
    log-likelihood of each answer choice.

    The model picks the answer that it finds most "natural"
    (highest probability) to follow the question.
    """
    answer_labels = ["A", "B", "C", "D"]
    best_score = float("-inf")
    best_answer = 0

    for i, choice in enumerate(choices):
        # Format as: "Question: ... Answer: A) ..."
        prompt = f"Question: {question}\nAnswer: {answer_labels[i]}) {choice}"
        inputs = tokenizer(prompt, return_tensors="pt")

        with torch.no_grad():
            outputs = model(**inputs, labels=inputs["input_ids"])
            # Lower loss = higher probability = model finds this more natural
            score = -outputs.loss.item()

        if score > best_score:
            best_score = score
            best_answer = i

    return best_answer


print("Answer function ready!")

In [ ]:
# Evaluate GPT-2 on a subset of questions (full eval would be slow)
NUM_QUESTIONS = 20  # Evaluate on 20 questions per subject

print(f"Evaluating GPT-2 on {NUM_QUESTIONS} questions per subject...")
print("=" * 60)

results_by_subject = {}

for subject_name, dataset in mmlu_data.items():
    # Take a sample of questions
    sample_size = min(NUM_QUESTIONS, len(dataset))
    indices = random.sample(range(len(dataset)), sample_size)

    model_answers = []
    correct_answers = []

    for idx in indices:
        q = dataset[idx]
        pred = gpt2_answer_mmlu(q["question"], q["choices"])
        model_answers.append(pred)
        correct_answers.append(q["answer"])

    result = evaluate_on_mmlu(model_answers, correct_answers)
    results_by_subject[subject_name] = result
    print(f"  {subject_name:25s}: {result['accuracy']:.1%} ({result['correct']}/{result['total']})")

# Overall
total_correct = sum(r["correct"] for r in results_by_subject.values())
total_questions = sum(r["total"] for r in results_by_subject.values())
overall = total_correct / total_questions if total_questions > 0 else 0

print(f"\n  {'OVERALL':25s}: {overall:.1%} ({total_correct}/{total_questions})")
print(f"\n  Random baseline: ~25%")
print(f"  GPT-2 (2019):    {overall:.1%}")
print(f"  GPT-4 (2023):    ~86%")
print(f"  Claude 3.5:      ~88%")
print(f"\nGPT-2 is small and old -- modern models are MUCH better!")

In [ ]:
# Visualize results by subject
subjects = list(results_by_subject.keys())
accuracies = [results_by_subject[s]["accuracy"] for s in subjects]

colors = ["#4CAF50" if a > 0.4 else "#FFC107" if a > 0.25 else "#F44336" for a in accuracies]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(subjects, accuracies, color=colors, edgecolor="black", linewidth=0.5)
ax.axhline(y=0.25, color="red", linestyle="--", linewidth=1, label="Random baseline (25%)")
ax.set_ylabel("Accuracy", fontsize=12)
ax.set_title("GPT-2 Performance on MMLU by Subject", fontsize=14, fontweight="bold")
ax.legend()
ax.set_ylim(0, 1.0)

# Add value labels
for bar, acc in zip(bars, accuracies):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
            f"{acc:.0%}", ha="center", fontweight="bold")

plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()

---
## 6. The Evolution of Benchmarks

Benchmarks get "solved" over time as models improve. Here's a visualization
of how AI performance has progressed on major benchmarks:

In [ ]:
# Approximate historical scores on major benchmarks
# (simplified for illustration)

benchmarks_history = {
    "GLUE": {
        "years": [2018, 2019, 2020, 2021, 2022],
        "scores": [69, 84, 89, 91, 92],
        "human": 87,
        "color": "#2196F3",
    },
    "SuperGLUE": {
        "years": [2019, 2020, 2021, 2022, 2023],
        "scores": [60, 78, 88, 91, 93],
        "human": 90,
        "color": "#FF9800",
    },
    "MMLU": {
        "years": [2021, 2022, 2023, 2024, 2025],
        "scores": [44, 58, 86, 88, 90],
        "human": 90,
        "color": "#4CAF50",
    },
}

fig, ax = plt.subplots(figsize=(12, 6))

for name, data in benchmarks_history.items():
    ax.plot(data["years"], data["scores"], "o-", linewidth=2,
            markersize=8, label=f"{name} (best AI)", color=data["color"])
    # Human baseline
    ax.axhline(y=data["human"], color=data["color"], linestyle=":", alpha=0.5)
    ax.text(data["years"][-1] + 0.1, data["human"],
            f"{name} human: {data['human']}%", fontsize=8, color=data["color"])

ax.set_xlabel("Year", fontsize=12)
ax.set_ylabel("Score (%)", fontsize=12)
ax.set_title("AI Progress on Major Benchmarks\n(approximate best model scores)",
             fontsize=14, fontweight="bold")
ax.legend(loc="lower right")
ax.set_ylim(30, 100)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Key observation: Models keep improving and eventually surpass human scores!")
print("When a benchmark is 'solved', researchers create a harder one.")
print("This is called the 'benchmark treadmill'.")

---
## 7. Comparing Benchmarks: What Does Each One Test?

Let's create a visual overview of the major benchmarks and what they measure.

In [ ]:
# Benchmark overview
benchmark_info = {
    "Benchmark": ["GLUE", "SuperGLUE", "MMLU", "HellaSwag", "HumanEval", "TruthfulQA", "BIG-Bench"],
    "Year": [2018, 2019, 2021, 2019, 2021, 2022, 2022],
    "Tests": [
        "Language understanding",
        "Harder language understanding",
        "Knowledge (57 subjects)",
        "Common sense reasoning",
        "Code generation",
        "Truthfulness",
        "200+ diverse tasks",
    ],
    "Size": ["9 tasks", "8 tasks", "~16K Q's", "~10K Q's", "164 problems", "817 Q's", "200+ tasks"],
    "Status": ["Solved", "Mostly solved", "Active", "High scores", "Active", "Active", "Active"],
}

# Print as a nice table
print(f"{'Benchmark':<15} {'Year':<6} {'Tests':<30} {'Size':<15} {'Status':<15}")
print("=" * 81)
for i in range(len(benchmark_info["Benchmark"])):
    print(f"{benchmark_info['Benchmark'][i]:<15} "
          f"{benchmark_info['Year'][i]:<6} "
          f"{benchmark_info['Tests'][i]:<30} "
          f"{benchmark_info['Size'][i]:<15} "
          f"{benchmark_info['Status'][i]:<15}")

---
## 8. Pitfalls: Why Benchmarks Aren't Perfect

In [ ]:
# Demonstration: Data contamination
# If a model has seen the benchmark questions during training,
# its score is artificially inflated

print("Pitfall: Data Contamination")
print("=" * 50)
print()

# Simulate a "cheating" model that memorized some answers
if mmlu_data:
    subject_name = list(mmlu_data.keys())[0]
    dataset = mmlu_data[subject_name]
    correct_answers = [q["answer"] for q in dataset][:20]

    # Honest model: random guessing
    honest = [random.randint(0, 3) for _ in correct_answers]
    honest_result = evaluate_on_mmlu(honest, correct_answers)

    # "Contaminated" model: knows 50% of answers from training
    contaminated = []
    for i, correct in enumerate(correct_answers):
        if random.random() < 0.5:  # 50% chance it saw this question
            contaminated.append(correct)  # Memorized answer!
        else:
            contaminated.append(random.randint(0, 3))  # Random guess
    contaminated_result = evaluate_on_mmlu(contaminated, correct_answers)

    print(f"  Honest model (random):      {honest_result['accuracy']:.1%}")
    print(f"  Contaminated model (50%):   {contaminated_result['accuracy']:.1%}")
    print()
    print("  The contaminated model looks better, but it's CHEATING!")
    print("  It memorized answers instead of actually learning.")
    print("  This is a real problem in AI evaluation.")

In [ ]:
# Demonstration: Benchmark score vs. real-world performance
print("Pitfall: High Benchmark Score != Good in Practice")
print("=" * 55)
print()
print("A model might score 90% on MMLU but still:")
print("  - Give unhelpful answers to real users")
print("  - Hallucinate (make up false information confidently)")
print("  - Fail on YOUR specific use case")
print("  - Be biased in ways the benchmark doesn't test")
print()
print("Lesson: Benchmarks are useful for comparison, but always")
print("test models on YOUR actual data and use cases too!")

---
## 9. Experiment: Create Your Own Mini-Benchmark!

Now it's your turn. Let's create a simple benchmark with your own questions
and test a model on it.

In [ ]:
# Create your own mini-benchmark!
# Format: (question, [choices], correct_answer_index)

my_benchmark = [
    {
        "question": "What is the capital of France?",
        "choices": ["London", "Paris", "Berlin", "Madrid"],
        "answer": 1,  # Paris (0-indexed)
    },
    {
        "question": "Which planet is closest to the Sun?",
        "choices": ["Venus", "Earth", "Mercury", "Mars"],
        "answer": 2,  # Mercury
    },
    {
        "question": "What is 7 x 8?",
        "choices": ["54", "56", "58", "64"],
        "answer": 1,  # 56
    },
    {
        "question": "Who wrote Romeo and Juliet?",
        "choices": ["Charles Dickens", "Jane Austen", "William Shakespeare", "Mark Twain"],
        "answer": 2,  # Shakespeare
    },
    {
        "question": "What gas do plants absorb from the air?",
        "choices": ["Oxygen", "Nitrogen", "Carbon Dioxide", "Hydrogen"],
        "answer": 2,  # CO2
    },
    # Add your own questions here!
]

print(f"My Mini-Benchmark: {len(my_benchmark)} questions")
print("=" * 50)

# Test GPT-2 on our custom benchmark
answer_labels = ["A", "B", "C", "D"]
model_answers = []
correct_answers = []

for q in my_benchmark:
    pred = gpt2_answer_mmlu(q["question"], q["choices"])
    model_answers.append(pred)
    correct_answers.append(q["answer"])

    correct_marker = "CORRECT" if pred == q["answer"] else "WRONG"
    print(f"\nQ: {q['question']}")
    print(f"  GPT-2 answered: {answer_labels[pred]}) {q['choices'][pred]}")
    print(f"  Correct answer: {answer_labels[q['answer']]}) {q['choices'][q['answer']]}")
    print(f"  Result: {correct_marker}")

result = evaluate_on_mmlu(model_answers, correct_answers)
print(f"\nOverall Score: {result['accuracy']:.0%} ({result['correct']}/{result['total']})")

---
## 10. Summary

```
┌───────────────────────────────────────────────────────────┐
│                 Benchmarks Cheat Sheet                     │
│                                                           │
│  What:     Standardized tests for fair model comparison   │
│  Why:      Compare models objectively with numbers        │
│  Key ones: MMLU (knowledge), HumanEval (coding),          │
│            HellaSwag (common sense), TruthfulQA (honesty) │
│                                                           │
│  Remember:                                                │
│    - Random baseline for 4-choice = 25%                   │
│    - Benchmarks get solved (treadmill effect)             │
│    - Watch for data contamination                         │
│    - High score doesn't mean good for YOUR task           │
│    - Always test on your own data too                     │
└───────────────────────────────────────────────────────────┘
```

### What We Covered

1. Loaded and explored real MMLU benchmark questions
2. Built a simple evaluator to score model answers
3. Tested GPT-2 on benchmark questions (and saw it struggle!)
4. Visualized the evolution of AI benchmark scores
5. Explored pitfalls: data contamination and benchmark limitations
6. Created our own mini-benchmark

### Next Steps

- Read the [Benchmarks guide](../benchmarks/README.md) for more details on each benchmark
- Check out [Hugging Face's Open LLM Leaderboard](https://huggingface.co/spaces/open-llm-leaderboard/open_llm_leaderboard) to see how models compare